In [209]:
import pandas as pd
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import os
import matplotlib as mpl
mpl.use('TkAgg') 
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
plt.ion()

In [210]:
def load_AllOfTheData():
    df = pd.read_csv('../Parsing/CSVDATAWITHDT.txt')
    
    return df

In [211]:
data = load_AllOfTheData()

data.head()

,'ThreatId','SourceSystem','ThreatName','Priority','Description','Affiliation','Location','Location.Latitude','Location.Longitude','Location.Altitude',...,'Tracks.Lob.OriginPosition.Lla.Altitude','Tracks.Lob.OriginPosition.Ecef','Tracks.Lob.OriginPosition.DataCase','Tracks.Feature.Importance.Importance','Countermeasures.DeviceId','Countermeasures.Countermeasure','Countermeasures.State','Tracks.Feature.Countermeasures.DroneModel','Countermeasures.DroneModel','DetectionTimeInDT'
0,3784,TPM,DON 3784,8,NaN,2,NaN,42.491578,-71.311929,1263.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:08:02.495332
1,3786,TPM,DON 3786,8,NaN,2,NaN,42.414375,-71.351943,10382.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:07:59.591863
2,3726,TPM,DON 3726,8,NaN,2,NaN,42.770994,-71.986791,5065.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:08:02.233450
3,3794,TPM,DON 3794,8,NaN,2,NaN,42.287262,-71.336343,3855.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:08:01.878965
4,3797,TPM,DON 3797,8,NaN,2,NaN,42.321132,-71.247895,4228.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:08:02.521955


In [212]:
data.columns = data.columns.str.strip() #There's extra white space in the data
data.columns = data.columns.str.replace("'", "") #The columns already contain quotes we need to remove 
#data.fillna('0', inplace=True)

ThreatId = data['ThreatId'].to_numpy()
Latitude = data['Location.Latitude'].to_numpy()
Longitude = data['Location.Longitude'].to_numpy()
Altitude = data['Location.Altitude'].to_numpy()
DetectionTime = data['DetectionTimeInDT'].to_numpy()


In [213]:
# create a dictionary and store each ThreatID in it, 
# if the Threat is already in the dictionary, append the current L,L,A coordinates
# we then can iterate through Dictionary and plot each threat ID

In [214]:
ThreatsXY = {}
ThreatsHT = {}

for i, threat_num in enumerate(ThreatId):


    latitude = Latitude[i]
    longitude = Longitude[i]
    altitude = Altitude[i]
    detection_time = DetectionTime[i] # string value 


    new_coordinate_xy = [latitude, longitude, detection_time] # first graph 
    new_coordinate_ht = [altitude, detection_time] # second graph 


    # populating dictionary for longitude vs latitude graph 
    # eliminating duplicates in dictionary without using a series  
    if threat_num in ThreatsXY: 
        if (new_coordinate_xy not in ThreatsXY[threat_num]):
            ThreatsXY[threat_num].append(new_coordinate_xy)
    
    else:
        ThreatsXY[threat_num] = [new_coordinate_xy]


    # populating dictionary for height vs time graph 
    # eliminating duplicates in dictionary without using a series  
    if threat_num in ThreatsHT: 
        if new_coordinate_ht not in ThreatsHT[threat_num]:
            ThreatsHT[threat_num].append(new_coordinate_ht)


    else: 
        ThreatsHT[threat_num] = [new_coordinate_ht]


# print both dictionarys 
print("XY Threats",  len(ThreatsXY[4069]))
print("HT Threats",  len(ThreatsHT[4069]))


XY Threats 156
HT Threats 156


In [215]:
threat_counts = data['ThreatId'].value_counts()

print(threat_counts)

ThreatId
3709    677
1006    677
3633    535
3801    489
3746    423
       ... 
4079      1
4096      1
3832      1
4147      1
4100      1
Name: count, Length: 226, dtype: int64


In [216]:
# h t graph 

# for the current id 
for id, coord in list(ThreatsHT.items())[0:4]: 
    graph_data = {"Height":[], "Time":[], "ThreatID":[]} 
    num_points = 0
    for i in range(len(coord)):
        graph_data['Height'].append(coord[i][0])
        graph_data['Time'].append(coord[i][1])

        num_points +=1
    
    # plt.figure(figsize=(15,8))
    # plt.plot(graph_data['Time'], graph_data['Height'], label=f"Number of hits: {num_points}" , linestyle='-', marker='x')
    # plt.xlabel('Time')
    # plt.xticks(rotation=45, ha='right')
    # plt.ylabel('Height')
    # plt.title(f'Flight Time for ID: {id}')
    # plt.legend()

    






In [217]:

# x y graph 

# for the current id 
for id, coord in list(ThreatsXY.items())[4:8]:
    graph_data = {"Longitude":[], "Latitude":[], "ThreatID":[]}
    for i in range(len(ThreatsXY[id])):
        # append coordinates to the dictionary lists
        graph_data["Longitude"].append(coord[i][0])
        graph_data["Latitude"].append(coord[i][1])

    # # plot scatter data
    # plt.figure(figsize=(10,6))
    # plt.title(f'Top level view of {id}')
    # plt.xlabel('Longitude')
    # plt.ylabel('Latitude')
    # plt.plot(graph_data["Latitude"], graph_data["Longitude"], linestyle='-', marker='x')


In [218]:
Detection_times = {}

# Fill detection_times dictionary to show,  {id: array_of_time_change_in_seconds}
for i, threat in enumerate(ThreatId):
    id = ThreatId[i]
    date = DetectionTime[i]
    date_split = date.split(':')
    seconds = date_split[2]
    if id not in Detection_times:
        Detection_times[id] = [float(seconds)]
    else:
        Detection_times[id].append(float(seconds))
#create new feature if Signal is persistent (detected frequently within 6 seconds)

data['Signal Persistence'] = False

#check if there is an time when the diff is less than or equal to 6 seconds, if so change feature to True
for id, seconds in Detection_times.items():
    for left in range(len(seconds)-5):
        right = left + 5
        diff = seconds[right] - seconds[left]
        if diff <= 6.0:
            data.loc[data['ThreatId'] == id, "Signal Persistence"] == True
            break



In [219]:
SlopeROC = {}

for threat_num in ThreatId:
    for i in range(0,len(ThreatsXY[threat_num])-1):

        detection_time = DetectionTime[i] # string value 

        point1 = ThreatsXY[threat_num][i]
        point2 = ThreatsXY[threat_num][i+1]
        m = (point2[1] - point1[1])/(point2[0]-point1[0])

        new_slope = [m, detection_time]

    # populating dictionary for longitude vs latitude graph 
    # eliminating duplicates in dictionary without using a series
        if threat_num in SlopeROC:  
            SlopeROC[threat_num].append(new_slope)
        else:
            SlopeROC[threat_num] = [new_slope]

C:\Users\nbohm\AppData\Local\Temp\ipykernel_29464\3206808834.py:10: RuntimeWarning: invalid value encountered in scalar divide
  m = (point2[1] - point1[1])/(point2[0]-point1[0])


In [220]:
#plotting the slope changes for the scatter data
# for the current id 
for id, coord in list(SlopeROC.items())[4:8]:
    graph_data = {"Slope":[], "Detection Time":[], "ThreatID":[]}
    for i in range(len(SlopeROC[id])):
        # append coordinates to the dictionary lists
        graph_data["Slope"].append(coord[i][0])
        graph_data["Detection Time"].append(coord[i][1])

    # # plot scatter data for slope changes
    # plt.figure(figsize=(20,8))
    # plt.title(f'Slopes of {id}')
    # plt.xlabel('Detection Time')
    # plt.ylabel('Slope')
    # plt.scatter(graph_data["Detection Time"], graph_data["Slope"], marker='.') 

In [251]:
IDofInterest = 4069

# x y graph 

# for the current id of interest
graph_data = {"Longitude":[], "Latitude":[], "ThreatID":[]}
for i in range(len(ThreatsXY[IDofInterest])):
    # append coordinates to the dictionary lists
    graph_data["Longitude"].append(ThreatsXY[IDofInterest][i][0])
    graph_data["Latitude"].append(ThreatsXY[IDofInterest][i][1])

# plot scatter data
# plt.figure(figsize=(10,6))
# plt.title(f'Top level view of {IDofInterest}')
# plt.xlabel('Longitude')
# plt.ylabel('Latitude')
# plt.plot(graph_data["Latitude"], graph_data["Longitude"], linestyle='-', marker='x')

# calculating the slope changes between points

SlopeROC = {}

for i in range(0,len(ThreatsXY[IDofInterest])-1):

    point1 = ThreatsXY[IDofInterest][i]
    point2 = ThreatsXY[IDofInterest][i+1]
    if point2[0] != point1[0]:
        m = (point2[1] - point1[1])/(point2[0]-point1[0])
        new_slope = [m, ThreatsXY[IDofInterest][i][2]]
    else:
        m = 0

    if IDofInterest in SlopeROC:  
        SlopeROC[IDofInterest].append(new_slope)
    else:
        SlopeROC[IDofInterest] = [new_slope]

#plotting the slope changes for the scatter data
graph_data = {"Slope":[], "Detection Time":[], "DeltaT":[0]}
for i in range(len(SlopeROC[IDofInterest])):
    # append coordinates to the dictionary lists
    if SlopeROC[IDofInterest][i][1] not in graph_data["Detection Time"]:
        graph_data["Slope"].append(SlopeROC[IDofInterest][i][0])
        graph_data["Detection Time"].append(SlopeROC[IDofInterest][i][1])

# plot scatter data for slope changes
# plt.figure(figsize=(20,8))
# plt.title(f'Slopes of {IDofInterest}')
# plt.xlabel('Detection Time')
# plt.ylabel('Slope')
# plt.scatter(graph_data["Detection Time"], graph_data["Slope"], marker='.') 

# plotting slopes in terms of seconds passed

from datetime import datetime
t0 = datetime.strptime(SlopeROC[IDofInterest][0][1],"%Y-%m-%d %H:%M:%S.%f")
print(t0)

for i in range(1,len(SlopeROC[IDofInterest])):

    ti = datetime.strptime(SlopeROC[IDofInterest][i][1],"%Y-%m-%d %H:%M:%S.%f")

    time_diff = ti - t0
    deltat = time_diff.total_seconds()

    if deltat not in graph_data["DeltaT"]:
        graph_data["DeltaT"].append(deltat)
    else:
        next

print(len(graph_data['DeltaT']))
print(len(graph_data['Slope']))

# plot scatter data for slope changes
plt.figure(figsize=(20,8))
plt.title(f'Slopes of {IDofInterest} from T0')
plt.xlabel('Tn - T0 (seconds)')
plt.ylabel('Slope')
plt.scatter(graph_data["DeltaT"], graph_data["Slope"], marker='.') 

#test commit

2022-07-13 15:15:54.115159
116
116


In [286]:
import numpy as np
from scipy.interpolate import BSpline
from scipy.interpolate import splrep, splev,CubicSpline
import matplotlib.pyplot as plt
 
#### this is avoiding the idea of knots 
# less specific to us but same idea

# Sample data
x = graph_data["DeltaT"]
x_array = np.array(graph_data["DeltaT"])
y = graph_data["Slope"]
y_array = np.array(graph_data["Slope"])

# Find spline representation
tck = CubicSpline(x, y)

# Evaluate spline at new points
xnew = np.linspace(0, 212, 1000)
ynew = tck(xnew)

# Plotting
plt.figure(figsize=(20,8))
plt.plot(xnew, ynew, c='r')
plt.scatter(x, y, c='g')
plt.title('Spline Interpolation')
plt.xlabel('x')
plt.ylabel('S(x)')
plt.show()